### **1. Problem Setup & Data Split**

Se predice 'quality' (variable objetivo, escala 3-8) del dataset *Red Wine Quality* (1599 muestras, 11 features fisicoquímicos). Se usa **un único split 80/20** con 'random_state' **fijo** para que todos los modelos se evalúen sobre exactamente el mismo test set. Esto garantiza una comparación justa.



In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

df = pd.read_csv("winequality-red.csv")
X = df.drop(columns="quality")
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

### **2. Evaluation Helper**

In [15]:
def evaluate(model, X_te, y_te, name):
    """Compute regression metrics on the test set.

    Args:
        model: Fitted estimator exposing a ``predict`` method.
        X_te: Test features.
        y_te: Test target.
        name: Label used in the comparison table.

    Returns:
        dict: Model name, RMSE, MAE and R2.
    """
    y_pred = model.predict(X_te)
    return {
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_te, y_pred)),
        "MAE": mean_absolute_error(y_te, y_pred),
        "R2": r2_score(y_te, y_pred),
    }

### **3. Model Training: Baselines vs. Ensemble**

El SVR se envuelve en un 'Pipeline' con 'StandardScaler' para evitar **data leakage**: el scaler se ajusta solo con 'X_train'. El árbol y el Random Forest no requieren feature scaling, ya que sus splits son invariantes a transformaciones monótonas.

In [16]:
tree = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)
svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0))
rf = RandomForestRegressor(
    n_estimators=300,
    max_features="sqrt",  # decorrelates trees (random feature subsets per split)
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

for model in (tree, svr, rf):
    model.fit(X_train, y_train)

results = pd.DataFrame([
    evaluate(tree, X_test, y_test, "Decision Tree"),
    evaluate(svr, X_test, y_test, "SVR (RBF)"),
    evaluate(rf, X_test, y_test, "Random Forest"),
]).round(4)
print(results)

           Model    RMSE     MAE      R2
0  Decision Tree  0.6584  0.4964  0.3366
1      SVR (RBF)  0.5928  0.4535  0.4623
2  Random Forest  0.5419  0.4150  0.5506


### **4. Key Insights**

* **Performance ranking**: Random Forest > SVR > Decision Tree en las tres métricas. El RMSE de 0.54 indica que, en promedio, la predicción se desvía ~0.5 puntos de una escala de calidad de 3 a 8.
* **Variance reduction**: el Random Forest promedia árboles entrenados sobre muestras bootstrap y subconjuntos aleatorios de features. Al decorrelacionar los árboles, el promedio reduce la varianza del árbol único sin aumentar mucho el bias.
* **R² interpretation**: el Random Forest explica ~55% de la varianza de quality. Hay margen de mejora, y el siguiente paso natural es el hyperparameter tuning (RandomizedSearchCV sobre max_features, min_samples_leaf, n_estimators).
* **Performance vs. interpretability trade-off**: el Random Forest es el mejor modelo en métricas, pero el menos interpretable. Un árbol de profundidad 5 es visualizable y explicable a negocio. El bosque requiere herramientas post-hoc como feature_importances_.
* **Caveat**: como el target es ordinal y discreto, una variante de clasificación u ordinal regression podría ser una alternativa válida.